# 04 — Anomaly Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/EnergySavvy-AI-Project/blob/main/notebooks/04_anomaly_detection.ipynb)


**Goal:** Detect unusual consumption relative to a defined or learned normal pattern.

**Input:** `data/household_power/processed/household_power_hourly.parquet` + `models/forecast_rf.pkl` (from 03)

**Output:** `reports/results/anomaly_report.csv` + `reports/figures/anomaly_*.png`

**Idea:** If forecasting model from 03 learns normal behavior, then large prediction errors = anomalies. We combine residual analysis +
Isolation Forest.

## 0. Setup & Load Data
Load hourly data. This cell is Colab-safe: it loads from GitHub raw if file not found locally.


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestRegressor
PROCESSED = Path("data/household_power/processed/household_power_hourly.parquet")
MODELS_DIR = Path("models")
RESULTS_DIR = Path("reports/results")
FIG_DIR = Path("reports/figures")
for d in [RESULTS_DIR, FIG_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# Colab-safe loading
if not PROCESSED.exists():
    URL = "https://raw.githubusercontent.com/Mohamed-Al-Saudi/EnergySavvy-AI-Project/main/data/household_power/processed/househ
    print(f"Processed not found locally, loading from GitHub raw...")
    df = pd.read_parquet(URL)
else:
    df = pd.read_parquet(PROCESSED)
df.index = pd.to_datetime(df.index)
print(f"Loaded: {df.shape}")
df.head()

## 1. Feature Engineering (Same as 03)
To reuse the forecasting model, we must recreate exactly the same features:
- **Time features:** hour, dayofweek, month, is_weekend
- **Lags:** lag_1 (1h ago), lag_24 (same hour yesterday), lag_168 (same hour last week)
- **Rolling:** roll_24_mean, roll_24_std (mean/std of last 24h)

This ensures our model predictions are consistent.

In [ ]:
target = Global_active_power
df_t = df[[target]].asfreq('H')
# time features
df_t['hour'] = df_t.index.hour
df_t['dayofweek'] = df_t.index.dayofweek
df_t['month'] = df_t.index.month
df_t['is_weekend'] = (df_t.index.dayofweek >= 5).astype(int)
# lag features - the memory of the model
for lag in [1,2,3,24,168]:
    df_t[f'lag_{lag}'] = df_t[target].shift(lag)
# rolling features
df_t['roll_24_mean'] = df_t[target].shift(1).rolling(24).mean()
df_t['roll_24_std'] = df_t[target].shift(1).rolling(24).std()
df_t = df_t.dropna()
X = df_t.drop(columns=[target])
y = df_t[target]
print(f"After feature engineering: {df_t.shape}")
print(f"Features: {X.columns.tolist()}")
df_t.head()

## 2. Load Forecasting Model from 03
The model learned normal consumption. If actual consumption is far from prediction, it's anomalous.
If `forecast_rf.pkl` not found (e.g. running in Colab without uploading model), we train a quick RF on 70% train

In [ ]:
model_path = MODELS_DIR / "forecast_rf.pkl"
try:
    rf = joblib.load(model_path)
    print(f"Loaded model: {model_path}")
except Exception as e:
    print(f"Model not found ({e}), training quick RF...")
    split = int(len(df_t)*0.7)
    rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[:split], y.iloc[:split])
# Predict normal behavior
df_t['y_pred'] = rf.predict(X)
df_t['residual'] = df_t[target] - df_t['y_pred']
df_t['abs_error'] = df_t['residual'].abs()
df_t[['Global_active_power','y_pred','residual']].tail()

## 3. Anomaly Strategy 1: Residual + Z-Score (Main Method)
**Logic:**
- `residual = y_true - y_pred`
- If model is good, residual ~ 0. Large residual = model surprised = anomaly.
- We compute rolling Z-score: z = (residual - rolling_mean) / rolling_std over last week (24*7 hours).
- Flag if: `|z| > 3` (3 sigma) OR `abs_error > 2 * RMSE`.

This is adaptive: threshold changes with volatility, not fixed kW.


In [ ]:
window = 24*7 # 1 week rolling window for adaptive threshold
df_t['resid_roll_mean'] = df_t['residual'].rolling(window).mean()
df_t['resid_roll_std'] = df_t['residual'].rolling(window).std()
df_t['z_score'] = (df_t['residual'] - df_t['resid_roll_mean']) / df_t['resid_roll_std']
# Robust threshold - use 90th percentile of abs_error instead of fixed value
RMSE_proxy = df_t['abs_error'].quantile(0.9)
df_t['is_anomaly_resid'] = (df_t['z_score'].abs() > 3.0) | (df_t['abs_error'] > 2*RMSE_proxy)
print(f"RMSE proxy (90th pct): {RMSE_proxy:.3f}")
print(f"Anomalies (residual method): {df_t['is_anomaly_resid'].sum()} / {len(df_t)} ({df_t['is_anomaly_resid'].mean()*100:.2f}%

## 4. Anomaly Strategy 2: Isolation Forest (Unsupervised Baseline)
Isolation Forest learns normal feature space without labels. It isolates outliers by randomly partitioning data. We set `contamination=0.02` = expect 2% anomalies.

Final anomaly flag = `residual OR isolation_forest` to catch both point anomalies (spikes) and contextual anomalies (e.g. high
consumption at 3 AM).

In [ ]:
iso = IsolationForest(contamination=0.02, random_state=42)
df_t['is_anomaly_iso'] = iso.fit_predict(X) == -1
print(f"Anomalies (IsolationForest): {df_t['is_anomaly_iso'].sum()}")
# Combined
df_t['is_anomaly'] = df_t['is_anomaly_resid'] | df_t['is_anomaly_iso']
print(f"Final anomalies (union): {df_t['is_anomaly'].sum()} / {len(df_t)} ({df_t['is_anomaly'].mean()*100:.2f}%)")

## 5. Visualization
1. Time series last 14 days: True vs Predicted with red dots = anomalies
2. Residual histogram: should be ~normal centered at 0

In [ ]:
# Plot last 14 days
plot_df = df_t.last('14D')
plt.figure(figsize=(14,4))
plt.plot(plot_df.index, plot_df[target], label='True', alpha=0.7)
plt.plot(plot_df.index, plot_df['y_pred'], label='Pred (normal)', alpha=0.7)
anoms = plot_df[plot_df['is_anomaly']]
plt.scatter(anoms.index, anoms[target], c='red', s=30, label='Anomaly', zorder=5)
plt.legend()
plt.title('Anomalies - Last 14 Days (True vs Expected)')
plt.tight_layout()
plt.savefig(FIG_DIR / "anomaly_14d.png", dpi=150)
plt.show()
# Residual histogram
plt.figure(figsize=(6,4))
plt.hist(df_t['residual'].dropna(), bins=100)
plt.title('Residual Distribution (y_true - y_pred)')
plt.xlabel('Residual (kW)')
plt.tight_layout()
plt.savefig(FIG_DIR / "anomaly_residual_hist.png", dpi=150)
plt.show()

## 6. Save Anomaly Report
Save sorted by absolute error (most severe first) for dashboard and for 05_recommendation.


In [ ]:
report = df_t[df_t['is_anomaly']][[target, 'y_pred', 'residual', 'z_score', 'abs_error', 'hour', 'dayofweek']].sort_values('abs
report.to_csv(RESULTS_DIR / "anomaly_report.csv")
print(f"Saved anomaly report: {len(report)} anomalies -> {RESULTS_DIR / 'anomaly_report.csv'}")
report.head(10)

## 7. Key findings for next phase

- **Anomaly rate ~2-4%** is normal for household data.

- **Residual method** catches point anomalies: sudden spikes where actual >> predicted (e.g., oven + AC left on).

- **Isolation Forest** catches contextual anomalies: high consumption at 3 AM (unusual hour).

- **Z-score > 3** is adaptive threshold, better than fixed kW threshold because it adjusts to volatility.

- **Next:** 05_recommendation_system.ipynb will use
residual and
hour to suggest savings (e.g., "Shift usage from 20h to 14h").